# Lab: pharmacogenomics from genotype to prescription

In this lab you work through the two halves of the chapter on real data and published numbers:

1. **Python** — translate *CYP2C19* genotypes into star alleles, diplotypes and metabolizer
   phenotypes, then apply a CPIC recommendation.
2. **Python / R** — compare pharmacogenomic allele frequencies between two Colombian populations,
   compute $F_{ST}$ yourself, and ask which variants would change a prescription.

Everything the lab needs is created in the notebook, so it runs anywhere Python and R are available.

```{admonition} Before you start
:class: tip
Work through {doc}`pharmagenes` and {doc}`knowledge-bases` first: you will need the allele function
terms and the metabolizer phenotype definitions.
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil

print("R  ", "found" if shutil.which("R") else "NOT found (the R section will not run)")

## Part 1 — Star alleles, diplotypes and phenotypes

A star allele is a **haplotype**: a combination of variants carried together on one copy of the gene.
Four *CYP2C19* alleles cover most of what is seen in practice. Each non-reference allele here is
defined by a single variant, which keeps the bookkeeping simple:

| Allele | Defining variant | Function |
|---|---|---|
| `*1` | none (reference) | normal |
| `*2` | rs4244285 (c.681G>A, splice defect) | no function |
| `*3` | rs4986893 (c.636G>A, premature stop) | no function |
| `*17` | rs12248560 (c.-806C>T, promoter) | increased |

Real allele definitions come from [PharmVar](https://www.pharmvar.org/), and most involve several
variants per haplotype.

In [ ]:
STAR = {
    "*1":  {"defining": None,                   "function": "normal"},
    "*2":  {"defining": ("rs4244285",  "A"),    "function": "no function"},
    "*3":  {"defining": ("rs4986893",  "A"),    "function": "no function"},
    "*17": {"defining": ("rs12248560", "T"),    "function": "increased"},
}

# activity score contributed by one allele
ACTIVITY = {"no function": 0.0, "decreased": 0.5, "normal": 1.0, "increased": 1.5}

Genotypes for a small clinic cohort. Each column is the unphased genotype at one variant.

In [ ]:
patients = pd.DataFrame([
    {"patient": "P01", "rs4244285": "GG", "rs4986893": "GG", "rs12248560": "CC"},
    {"patient": "P02", "rs4244285": "GA", "rs4986893": "GG", "rs12248560": "CC"},
    {"patient": "P03", "rs4244285": "AA", "rs4986893": "GG", "rs12248560": "CC"},
    {"patient": "P04", "rs4244285": "GG", "rs4986893": "GG", "rs12248560": "CT"},
    {"patient": "P05", "rs4244285": "GG", "rs4986893": "GG", "rs12248560": "TT"},
    {"patient": "P06", "rs4244285": "GA", "rs4986893": "GA", "rs12248560": "CC"},
    {"patient": "P07", "rs4244285": "GA", "rs4986893": "GG", "rs12248560": "CT"},
]).set_index("patient")
patients

### Calling star alleles

Count how many copies of each defining variant a person carries; every chromosome without a defining
variant is `*1`.

```{admonition} A simplification worth knowing about
:class: warning
These genotypes are **unphased**, so with defining variants at two different alleles we cannot be sure
which variant sits on which chromosome. Patient P07 is the interesting case. Real star-allele calling
uses phased data or population haplotype frequencies; tools such as PharmCAT do this properly.
```

In [ ]:
def call_diplotype(row):
    """Return the two star alleles carried by one person, as a sorted list."""
    haps = []
    for star, spec in STAR.items():
        if spec["defining"] is None:
            continue
        rsid, alt = spec["defining"]
        haps += [star] * row[rsid].count(alt)
    haps += ["*1"] * (2 - len(haps))          # the rest are reference
    return sorted(haps[:2], key=lambda s: int(s[1:]))


def phenotype(diplotype):
    """CPIC activity-score rules for CYP2C19."""
    score = sum(ACTIVITY[STAR[a]["function"]] for a in diplotype)
    if score >= 3.0: return "Ultrarapid metabolizer"
    if score > 2.0:  return "Rapid metabolizer"
    if score == 2.0: return "Normal metabolizer"
    if score >= 1.0: return "Intermediate metabolizer"
    return "Poor metabolizer"


calls = patients.apply(call_diplotype, axis=1)
report = pd.DataFrame({
    "diplotype": calls.map("/".join),
    "phenotype": calls.map(phenotype),
})
report

### Applying a guideline

*CYP2C19* and **clopidogrel** is the classic prodrug pairing: clopidogrel is inactive until CYP2C19
converts it, so a poor metabolizer generates too little active drug and is at risk of a further
cardiovascular event — the opposite of the efavirenz case in {doc}`cyp2b6-efavirenz`, where the
enzyme *inactivates* the drug.

In [ ]:
CLOPIDOGREL = {
    "Ultrarapid metabolizer":    "Standard dosing",
    "Rapid metabolizer":         "Standard dosing",
    "Normal metabolizer":        "Standard dosing",
    "Intermediate metabolizer":  "Consider an alternative antiplatelet (prasugrel, ticagrelor)",
    "Poor metabolizer":          "Avoid clopidogrel; use an alternative antiplatelet",
}
report["clopidogrel"] = report["phenotype"].map(CLOPIDOGREL)
report

```{admonition} Exercise 1
:class: note
1. Which patients would have their prescription changed by this result, and which would not?
2. Patient P05 is an ultrarapid metabolizer. For clopidogrel that is not a problem — but *CYP2C19*
   also metabolizes the proton-pump inhibitor omeprazole, which it **inactivates**. What would you
   expect for P05 on omeprazole, and why is the direction opposite to clopidogrel?
3. Add a patient who is `*2/*17` and work out their phenotype by hand before running the code.
   Does the activity-score rule give the answer you expected? (This diplotype is genuinely
   awkward, and CPIC has changed its guidance on it.)
```

## Part 2 — Pharmacogenomic variation between populations

Now the population half of the chapter. These are the published effect allele frequencies for
ancestry-associated pharmaSNPs in Antioquia and Chocó, from Table 1 of
[Nagar et al. (2019)](https://doi.org/10.3389/fgene.2019.00241).

In [ ]:
t1 = pd.DataFrame(
    [("rs1954787",  "Antidepressant efficacy",      0.62, 0.21),
     ("rs7793837",  "Salbutamol efficacy",          0.69, 0.24),
     ("rs11615",    "Platin toxicity",              0.48, 0.07),
     ("rs3812718",  "Antiepileptic resistance",     0.55, 0.27),
     ("rs1719247",  "Simvastatin adverse reaction", 0.54, 0.27),
     ("rs4149056",  "Simvastatin toxicity",         0.18, 0.05),
     ("rs4244285",  "Clopidogrel efficacy",         0.90, 0.84),
     ("rs1799853",  "Warfarin dosage",              0.88, 0.97),
     ("rs9923231",  "Warfarin dosage",              0.57, 0.88),
     ("rs6977820",  "Antipsychotic drug toxicity",  0.25, 0.68),
     ("rs2740574",  "Tacrolimus metabolism",        0.10, 0.59),
     ("rs11212617", "Metformin efficacy",           0.33, 0.74)],
    columns=["rsid", "drug", "antioquia", "choco"],
)
t1

### Computing $F_{ST}$

For one SNP in two populations with allele frequencies $p_1$ and $p_2$:

$$
H_S = \frac{2p_1(1-p_1) + 2p_2(1-p_2)}{2}, \qquad
H_T = 2\bar{p}(1-\bar{p}), \qquad
F_{ST} = \frac{H_T - H_S}{H_T}
$$

$H_S$ is the average heterozygosity **within** populations and $H_T$ the heterozygosity of the pooled
population. $F_{ST}$ is the share of the total that sits **between** them.

In [ ]:
def fst(p1, p2):
    """Fixation index for one biallelic SNP in two populations."""
    p1, p2 = np.asarray(p1, float), np.asarray(p2, float)
    hs = (2 * p1 * (1 - p1) + 2 * p2 * (1 - p2)) / 2
    pbar = (p1 + p2) / 2
    ht = 2 * pbar * (1 - pbar)
    return np.where(ht > 0, (ht - hs) / ht, 0.0)


# check the function against the two worked examples in the chapter figure
print("p = 0.5 / 0.4 ->", round(float(fst(0.5, 0.4)), 2), "  (chapter figure: 0.01)")
print("p = 0.9 / 0.2 ->", round(float(fst(0.9, 0.2)), 2), "  (chapter figure: 0.49)")

In [ ]:
t1["delta"] = (t1["antioquia"] - t1["choco"]).abs()
t1["fst"] = fst(t1["antioquia"], t1["choco"])
t1.sort_values("fst", ascending=False)

```{admonition} Exercise 2
:class: note
1. The chapter reports an average $F_{ST}$ of **0.07** for pharmaSNPs genome-wide. The mean across
   these twelve variants is roughly twice that. Why is that not a contradiction? (Look again at how
   this table was selected.)
2. rs4244285 differs by only 0.06 between the two populations, while rs2740574 differs by 0.49.
   Which of the two is more likely to change a prescribing decision, and does $F_{ST}$ or the raw
   frequency difference answer that question better?
3. Two variants have nearly the same $|\Delta p|$ but different $F_{ST}$. Explain how that happens.
   (Compare a pair near $p = 0.5$ with a pair near $p = 0.9$.)
```

### Plotting the contrast

In [ ]:
rows = t1.sort_values("delta")
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(rows))
ax.hlines(y, rows["choco"], rows["antioquia"], color="#A0A0A0", lw=2.5, zorder=1)
ax.scatter(rows["antioquia"], y, s=90, color="#1F6F80", zorder=2, label="Antioquia")
ax.scatter(rows["choco"], y, s=90, color="#6B3FA0", zorder=2, label="Chocó")
ax.set_yticks(y)
ax.set_yticklabels(rows["rsid"])
ax.set_xlabel("Effect allele frequency")
ax.set_xlim(0, 1)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, ncol=2, loc="lower right")
plt.tight_layout()

### The same comparison in R

If `rpy2` is available, the magic below runs R in this notebook. The point is to check your
$F_{ST}$ function against an independent implementation of the same formula.

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R -i t1
fst_r <- function(p1, p2) {
  hs <- (2*p1*(1-p1) + 2*p2*(1-p2)) / 2
  pbar <- (p1 + p2) / 2
  ht <- 2*pbar*(1-pbar)
  (ht - hs) / ht
}
t1$fst_r <- fst_r(t1$antioquia, t1$choco)
print(head(t1[order(-t1$fst_r), c('rsid', 'drug', 'antioquia', 'choco', 'fst_r')], 5))
cat('max |difference| vs Python:', max(abs(t1$fst_r - t1$fst)), '\n')

```{admonition} Exercise 3 — putting it together
:class: note
A health service covering both departments wants a single pharmacogenomic panel: one short list of
variants to type in every patient before prescribing.

1. Using the table above, choose the variants you would include, and say what rule you used.
2. Would you recommend the **same** panel in both departments? Justify the answer with numbers,
   not just with the ancestry difference.
3. {doc}`from-variants-to-assays` reports PCR concordance of 97.7–100% against exome genotypes.
   For a panel of the size you chose, how many patients would you expect to be miscalled per
   thousand tested at 97.7% accuracy per variant? What does that imply about which variants are
   safe to put on a cheap assay?
```

---

**Sources.** Allele frequencies are from Table 1 of
[Nagar SD et al. (2019) *Front Genet* 10:241](https://doi.org/10.3389/fgene.2019.00241)
(CC BY 4.0). Star-allele definitions follow [PharmVar](https://www.pharmvar.org/); phenotype and
prescribing rules follow [CPIC](https://cpicpgx.org/). The clinic cohort in Part 1 is invented for
the lab.